## cv2_canny_edge detector

In [8]:
import numpy as np
import cv2

In [9]:
def detect_edges(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Canny(gray, 100, 200)


## manual_canny_edge_detector

In [10]:
def calculate_gradients(gray):
    # Estimate derivatives directly with neighboring pixel differences.
    gray_float = gray.astype(np.float64)
    gx = np.zeros_like(gray_float)
    gy = np.zeros_like(gray_float)

    gx[:, 1:-1] = (gray_float[:, 2:] - gray_float[:, :-2]) / 2
    gy[1:-1, :] = (gray_float[2:, :] - gray_float[:-2, :]) / 2

    strength = np.hypot(gx, gy)
    angle = (np.degrees(np.arctan2(gy, gx)) + 180) % 180

    if strength.max() > 0:
        strength = strength / strength.max() * 255

    return strength, angle


In [11]:
def thin_edges(strength, angle):
    # Non-maximum suppression: keep a pixel if it is strongest along its direction.
    padded = np.pad(strength, 1)
    center = padded[1:-1, 1:-1]

    left, right = padded[1:-1, :-2], padded[1:-1, 2:]
    up, down = padded[:-2, 1:-1], padded[2:, 1:-1]
    up_right, down_left = padded[:-2, 2:], padded[2:, :-2]
    up_left, down_right = padded[:-2, :-2], padded[2:, 2:]

    horizontal = (angle < 22.5) | (angle >= 157.5)
    diagonal_down = (angle >= 22.5) & (angle < 67.5)
    vertical = (angle >= 67.5) & (angle < 112.5)
    diagonal_up = (angle >= 112.5) & (angle < 157.5)

    keep = np.zeros_like(strength, dtype=bool)
    keep |= horizontal & (center >= left) & (center >= right)
    keep |= diagonal_down & (center >= up_right) & (center >= down_left)
    keep |= vertical & (center >= up) & (center >= down)
    keep |= diagonal_up & (center >= up_left) & (center >= down_right)

    return np.where(keep, strength, 0)


In [12]:
def connect_edges(thin, low=50, high=100):
    # Strong pixels are kept; weak pixels are kept only if connected to a strong one.
    strong = thin >= high
    weak = (thin >= low) & (thin < high)
    edges = strong.copy()
    stack = list(zip(*np.nonzero(strong)))

    while stack:
        y, x = stack.pop()

        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                ny, nx = y + dy, x + dx
                if (0 <= ny < edges.shape[0] and
                    0 <= nx < edges.shape[1] and
                    weak[ny, nx] and not edges[ny, nx]):
                    edges[ny, nx] = True
                    stack.append((ny, nx))

    return (edges * 255).astype(np.uint8)


In [13]:
def detect_edges_manual(frame, low=50, high=100):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 1.4)

    strength, angle = calculate_gradients(gray)
    thin = thin_edges(strength, angle)
    return connect_edges(thin, low, high)


In [14]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    cap.release()
    raise RuntimeError("Could not open the camera. Check that it is connected and available.")

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("Could not read a frame from the camera.")
            break

        frame = cv2.flip(frame, 1)  # Mirror the camera view

        # Smaller frames keep the manual Python steps responsive.
        frame = cv2.resize(frame, (320, 240))
        opencv_result = detect_edges(frame)
        manual_result = detect_edges_manual(frame)

        cv2.imshow("OpenCV Canny", opencv_result)
        cv2.imshow("Manual Canny", manual_result)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
